Extract Item 7, MD&A, from your 100- 10-Ks. Put MD&A in your dataframe in one column. Run all of your code on the MD&A and put in data frame (your dataframe will contain data for the entire 10-K, and data for the MD&A)

Basically doubling the previous df bc adding the MD&A section now
Save your dataframe as a pickle object so you don't have to start from scratch anymore!!

make sure to clean the 10-K file


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install syllables
import nltk
nltk.download('punkt_tab')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 37.3 MB/s eta 0:00:00
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 9.0.1
    Uninstalling importlib_metadata-9.0.1:
      Successfully uninstalled importlib_metadata-9.0.1


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import syllables
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from bs4 import BeautifulSoup

In [ ]:
cmufile = "/content/drive/MyDrive/Fall 2026/Text Mining/Fall 2026 Text Mining/week 04/cmudict.txt"
with open (cmufile, encoding ='latin-1') as cmu:
  cmuLines=cmu.readlines()
syllableDict = {}
for aline in cmuLines[56:]:
  syllableCount =0
  for char in aline:
    if char.isdigit():
      syllableCount +=1
  syllableDict[aline.split()[0].lower()]=syllableCount

def getSyllables(word):
  syllables = syllableDict[word]
  return syllables

def isComplexWord(word):
  syllables = syllableDict[word]
  if syllables > 2:
    return True
  else:
    return False

def isFuture(sentence):
  futureWords = ['will', 'shall', 'may','might', 'should', 'going']
  sentWords = word_tokenize(sentence.lower())
  for word in futureWords:
    if word in sentWords:
        return True
  if 'gon' in sentWords and 'na' in sentWords:
    return True
  if 'going' in sentWords and 'to' in sentWords:
    return True

  return False


In [ ]:
#testing to see if gonna and going to are future
text = "NLTK is gonna to be a great NLP toolkit. It is going to make processing text easy!"
sentences = sent_tokenize(text)
words = word_tokenize(sentences[0])
print(words)
print(isFuture(sentences[0]))

words = word_tokenize(sentences[1])
print(words)
print(isFuture(sentences[1]))


['NLTK', 'is', 'gon', 'na', 'to', 'be', 'a', 'great', 'NLP', 'toolkit', '.']
True
['It', 'is', 'going', 'to', 'make', 'processing', 'text', 'easy', '!']
True


In [ ]:
folder_path ='/content/drive/MyDrive/Fall 2026/Text Mining/10-Ks Assignment'
print (os.listdir(folder_path))
print ("Number of files:", len(os.listdir(folder_path)))
rows = []

['10-K_1770787_2025-02-13.txt', '10-K_1690080_2025-03-31.txt', '10-K_1599407_2025-03-31.txt', '10-K_34782_2025-02-18.txt', '10-K_1347858_2025-03-20.txt', '10-K_1950976_2025-03-03.txt', '10-K_1860782_2025-03-25.txt', '10-K_1890702_2025-03-07.txt', '10-K_1948175_2025-03-10.txt', '10-K_933972_2025-03-28.txt', '10-K_910638_2025-03-27.txt', '10-K_66740_2025-02-05.txt', '10-K_1650648_2025-02-28.txt', '10-K_1946563_2025-03-27.txt', '10-K_1785173_2025-02-27.txt', '10-K_1555279_2025-03-07.txt', '10-K_1580808_2025-02-25.txt', '10-K_824142_2025-02-27.txt', '10-K_1876255_2025-03-26.txt', '10-K_1634452_2025-03-31.txt', '10-K_1982701_2025-03-31.txt', '10-K_1800_2025-02-21.txt', '10-K_1018840_2025-03-31.txt', '10-K_934549_2025-03-17.txt', '10-K_1070494_2025-02-27.txt', '10-K_899629_2025-02-14.txt', '10-K_843006_2025-03-25.txt', '10-K_1962918_2025-03-19.txt', '10-K_949858_2025-03-11.txt', '10-K_935036_2025-02-27.txt', '10-K_1680062_2025-03-03.txt', '10-K_2098_2025-03-06.txt', '10-K_715579_2025-03-14.t

In [ ]:
#dictionary
def load_dictionary(filepath):
  with open (filepath, 'r', encoding = 'utf-8') as f:
    return set(f.read().lower().split())
#folders
dictionary_folder = '/content/drive/MyDrive/Fall 2026/Text Mining/Dictionaries/'
negative_words = load_dictionary(dictionary_folder + "negative words.txt")
positive_words = load_dictionary(dictionary_folder + "positive words.txt")
litigious_words = load_dictionary(dictionary_folder + "litigious words.txt")

for filename in os.listdir(folder_path):
  if filename.endswith (".txt"):
    filepath = os.path.join (folder_path, filename)

  with open(filepath, 'r', encoding = 'utf-8') as f:
    raw_html = f.read()
    clean_text = BeautifulSoup(raw_html, 'lxml').get_text()
    clean_text = clean_text.replace('\xa0', ' ')

    sentences = sent_tokenize (clean_text)
    total_sentences = len (sentences)
    future_sentences = []
    for sentence in sentences:
      if isFuture(sentence):
        future_sentences.append(sentence)
    total_future_sentences = len(future_sentences)

    words = word_tokenize(clean_text.lower())
    words = [word for word in words if word.isalpha()]
    word_count = len(words)

    pos_count = sum(1 for w in words if w in positive_words)
    neg_count = sum(1 for w in words if w in negative_words)
    lit_count = sum(1 for w in words if w in litigious_words)

    pos_rate = pos_count / word_count if word_count > 0 else 0
    neg_rate = neg_count / word_count if word_count > 0 else 0
    lit_rate = lit_count / word_count if word_count > 0 else 0

    future_text = " ".join(future_sentences)
    future_words = word_tokenize(future_text.lower())
    future_words = [word for word in future_words if word.isalpha()]
    future_word_count = len(future_words)

    future_pos_count = sum(1 for w in future_words if w in positive_words)
    future_neg_count = sum(1 for w in future_words if w in negative_words)
    future_lit_count = sum(1 for w in future_words if w in litigious_words)

    future_pos_rate = future_pos_count / future_word_count if future_word_count > 0 else 0
    future_neg_rate = future_neg_count / future_word_count if future_word_count > 0 else 0
    future_lit_rate = future_lit_count / future_word_count if future_word_count > 0 else 0

    complex_count = 0
    for word in words:
      if word in syllableDict:
        if isComplexWord(word):
          complex_count +=1

    future_complex_count = 0
    for word in future_words:
      if word in syllableDict:
        if isComplexWord(word):
          future_complex_count += 1
    if total_sentences > 0 and word_count > 0:
      fog = 0.4* ((word_count/total_sentences)+ 100*(complex_count / word_count))
    else:
      fog = 0

    if total_future_sentences > 0 and future_word_count >0:
      future_fog = 0.4 * ((future_word_count / total_future_sentences))
    else:
      future_fog = 0
    matches = re.findall(
      r'ITEM 7\..*?(?=ITEM 7A\.)',
      clean_text,
      re.IGNORECASE | re.DOTALL
    )
    if matches:
      mda_text = max(matches, key=len)
    else:
      mda_text = ""
    mdasentences = sent_tokenize(mda_text)
    mdatotal_sentences = len(mdasentences)
    mdafuture_sentences = []
    for sentence in mdasentences:
      if isFuture(sentence):
        mdafuture_sentences.append(sentence)
    mdatotal_future_sentences = len(mdafuture_sentences)

    mdawords = word_tokenize(mda_text.lower())
    mdawords = [word for word in mdawords if word.isalpha()]
    mdaword_count = len(mdawords)

    mdapos_count = sum(1 for w in mdawords if w in positive_words)
    mdaneg_count = sum(1 for w in mdawords if w in negative_words)
    mdalit_count = sum(1 for w in mdawords if w in litigious_words)

    mdapos_rate = mdapos_count / mdaword_count if mdaword_count > 0 else 0
    mdaneg_rate = mdaneg_count / mdaword_count if mdaword_count > 0 else 0
    mdalit_rate = mdalit_count / mdaword_count if mdaword_count > 0 else 0

    mdafuture_text = " ".join(mdafuture_sentences)
    mdafuture_words = word_tokenize(mdafuture_text.lower())
    mdafuture_words = [word for word in mdafuture_words if word.isalpha()]
    mdafuture_word_count = len(mdafuture_words)

    mdafuture_pos_count = sum(1 for w in mdafuture_words if w in positive_words)
    mdafuture_neg_count = sum(1 for w in mdafuture_words if w in negative_words)
    mdafuture_lit_count = sum(1 for w in mdafuture_words if w in litigious_words)

    mdafuture_pos_rate = mdafuture_pos_count / mdafuture_word_count if mdafuture_word_count > 0 else 0
    mdafuture_neg_rate = mdafuture_neg_count / mdafuture_word_count if mdafuture_word_count > 0 else 0
    mdafuture_lit_rate = mdafuture_lit_count / mdafuture_word_count if mdafuture_word_count > 0 else 0

    mdacomplex_count = 0
    for word in mdawords:
      if word in syllableDict:
        if isComplexWord(word):
          mdacomplex_count +=1

    mdafuture_complex_count = 0
    for word in mdafuture_words:
      if word in syllableDict:
        if isComplexWord(word):
          mdafuture_complex_count += 1

    if mdatotal_sentences > 0 and mdaword_count > 0:
      mdafog = 0.4 * ((mdaword_count / mdatotal_sentences) + 100 * (mdacomplex_count / mdaword_count))
    else:
      mdafog = 0

    if mdatotal_future_sentences > 0 and mdafuture_word_count > 0:
      mdafuture_fog = 0.4 * ((mdafuture_word_count / mdatotal_future_sentences))
    else:
      mdafuture_fog = 0

  row = {
        'Filename': filename,
        'TotalSentences': total_sentences,
        'WordCount': word_count,
        'PositiveWordCount': pos_count,
        'NegativeWordCount': neg_count,
        'LitigiousWordCount': lit_count,
        'PositivityRate': pos_rate,
        'NegativitiyRate' : neg_rate,
        'LitigiousRate': lit_rate,
        'FuturePositiveWordCount' :future_pos_count,
        'FutureNegativeWordCount' : future_neg_count,
        'FutureLitigiousWordCount' : future_lit_count,
        'FuturePositivityRate': future_pos_rate,
        'FutureNegativityRate' : future_neg_rate,
        'FutureLitigiousRate': future_lit_rate,
        'ComplexWordCount': complex_count,
        'FOG': fog,
        'FutureFOG': future_fog,
        'FutureTotalSentences': total_future_sentences,
        'FutureWordCount': future_word_count,
        'FutureComplexWordCount' : future_complex_count,
        'MDAText': mda_text,
        'mdaTotalSentences': mdatotal_sentences,
        'mdaWordCount': mdaword_count,
        'mdaPositiveWordCount': mdapos_count,
        'mdaNegativeWordCount': mdaneg_count,
        'mdaLitigiousWordCount': mdalit_count,
        'mdaPositivityRate': mdapos_rate,
        'mdaNegativitiyRate' : mdaneg_rate,
        'mdaLitigiousRate': mdalit_rate,
        'mdaFuturePositiveWordCount' :mdafuture_pos_count,
        'mdaFutureNegativeWordCount' : mdafuture_neg_count,
        'mdaFutureLitigiousWordCount' : mdafuture_lit_count,
        'mdaFuturePositivityRate': mdafuture_pos_rate,
        'mdaFutureNegativityRate' : mdafuture_neg_rate,
        'mdaFutureLitigiousRate': mdafuture_lit_rate,
        'mdaComplexWordCount': mdacomplex_count,
        'mdaFOG': mdafog,
        'mdaFutureFOG': mdafuture_fog,
        'mdaFutureTotalSentences': mdatotal_future_sentences,
        'mdaFutureWordCount': mdafuture_word_count,
        'mdaFutureComplexWordCount' : mdafuture_complex_count
  }
  rows.append(row)

df=pd.DataFrame(rows)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

display (df)

df.to_pickle('/content/drive/MyDrive/Fall 2026/Text Mining/Data Frame/dataFrame.pk1')

df.to_excel('/content/drive/MyDrive/Fall 2026/Text Mining/Data Frame/dataFrame.xlsx', index=False)





,Filename,TotalSentences,WordCount,PositiveWordCount,NegativeWordCount,LitigiousWordCount,PositivityRate,NegativitiyRate,LitigiousRate,FuturePositiveWordCount,FutureNegativeWordCount,FutureLitigiousWordCount,FuturePositivityRate,FutureNegativityRate,FutureLitigiousRate,ComplexWordCount,FOG,FutureFOG,FutureTotalSentences,FutureWordCount,FutureComplexWordCount,MDAText,mdaTotalSentences,mdaWordCount,mdaPositiveWordCount,mdaNegativeWordCount,mdaLitigiousWordCount,mdaPositivityRate,mdaNegativitiyRate,mdaLitigiousRate,mdaFuturePositiveWordCount,mdaFutureNegativeWordCount,mdaFutureLitigiousWordCount,mdaFuturePositivityRate,mdaFutureNegativityRate,mdaFutureLitigiousRate,mdaComplexWordCount,mdaFOG,mdaFutureFOG,mdaFutureTotalSentences,mdaFutureWordCount,mdaFutureComplexWordCount
0,10-K_1770787_2025-02-13.txt,47176,588002,770,4729,5594,0.001310,0.008042,0.009514,346,1692,1047,0.006300,0.030810,0.019065,67828,9.599737,23.774026,924,54918,12673,Item 7. Management’s Discussion and Analysis o...,167,6007,29,56,14,0.004828,0.009322,0.002331,7,20,2,0.004636,0.013245,0.001325,1568,24.829176,20.133333,30,1510,398
1,10-K_1690080_2025-03-31.txt,15014,387953,930,5317,6401,0.002397,0.013705,0.016499,413,2119,1041,0.005026,0.025787,0.012668,79849,18.568620,23.545272,1396,82173,18851,Item 7. Management’s\nDiscussion and Analysis ...,175,5806,35,67,32,0.006028,0.011540,0.005512,23,21,4,0.015657,0.014295,0.002723,1589,24.218153,14.331707,41,1469,409
2,10-K_1599407_2025-03-31.txt,15473,451910,985,5132,6784,0.002180,0.011356,0.015012,361,1990,1206,0.003531,0.019464,0.011796,96237,20.200789,30.092421,1359,102239,22945,ITEM 7.MANAGEMENT’S DISCUSSION AND ANALYSIS OF...,255,7603,23,75,50,0.003025,0.009865,0.006576,8,22,31,0.003895,0.010711,0.015093,1921,22.032811,19.561905,42,2054,508
3,10-K_34782_2025-02-18.txt,20719,536938,646,4291,6282,0.001203,0.007992,0.011700,87,487,188,0.002558,0.014319,0.005528,86263,16.792390,32.010353,425,34011,5950,Item 7. Management’s Discussion and Analysis o...,348,11068,64,266,46,0.005782,0.024033,0.004156,10,36,17,0.006793,0.024457,0.011549,2892,23.173592,20.303448,29,1472,422
4,10-K_1347858_2025-03-20.txt,9783,362287,627,4911,6197,0.001731,0.013556,0.017105,169,843,735,0.003915,0.019531,0.017028,79271,23.565208,31.563437,547,43163,9522,Item 7.Management’s Discussion and Analysis of...,90,4107,15,95,22,0.003652,0.023131,0.005357,2,12,2,0.004202,0.025210,0.004202,961,27.612963,21.155556,9,476,135
5,10-K_1950976_2025-03-03.txt,10439,323961,917,3059,4345,0.002831,0.009442,0.013412,390,1148,515,0.005537,0.016300,0.007312,62717,20.157260,21.134284,1333,70430,15517,Item 7.\nManagement’s Discussion and Analysis ...,2,13,0,0,0,0.000000,0.000000,0.000000,0,0,0,0.000000,0.000000,0.000000,6,21.061538,0.000000,0,0,0
6,10-K_1860782_2025-03-25.txt,10716,361241,2208,4921,6494,0.006112,0.013622,0.017977,605,1569,1420,0.008762,0.022722,0.020565,81961,22.559666,21.461072,1287,69051,17495,Item 7. Management’s Discussion and Analysis o...,226,9465,141,123,78,0.014897,0.012995,0.008241,22,21,21,0.010023,0.009567,0.009567,2483,27.245609,21.414634,41,2195,581
7,10-K_1890702_2025-03-07.txt,1151,106998,49,441,640,0.000458,0.004122,0.005981,0,27,21,0.000000,0.004198,0.003265,36373,50.781999,98.938462,26,6431,2751,Item 7. Management’s Discussion and Analysis ...,3,14,0,1,0,0.000000,0.071429,0.000000,0,0,0,0.000000,0.000000,0.000000,7,21.866667,0.000000,0,0,0
8,10-K_1948175_2025-03-10.txt,1084,92176,44,300,501,0.000477,0.003255,0.005435,1,19,11,0.000154,0.002918,0.001689,30463,47.232777,108.533333,24,6512,2866,Item 7. Management’s Discussion and Analysis ...,3,14,0,1,0,0.000000,0.071429,0.000000,0,0,0,0.000000,0.000000,0.000000,7,21.866667,0.000000,0,0,0
9,10-K_933972_2025-03-28.txt,64998,710348,845,2398,4036,0.001190,0.003376,0.005682,277,1044,539,0.006392,0.024090,0.012437,54484,7.439524,23.521303,737,43338,9549,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,123,3464,24,52,51,0.006928,0.015012,0.014723,4,8,7,0.008677,0.017354,0.015184,895,21.599914,

In [ ]:
import re

testString1 = "blah blah blah Item 7.    Management’s Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read...\nItem 7A.    Quantitative and Qualitative Disclosures About Market Risk"

item7RegEx = re.compile("I(tem 7(.+?)Item 7A)", re.S)
#print(item7RegEx.findall(testString1))
print(item7RegEx.search(testString1).group(1))

In [ ]:
import re


In [ ]:
df = pd.read_pickle('/content/drive/MyDrive/Fall 2026/Text Mining/Data Frame/dataFrame.pkl')
display (df)

when you run one of the fuctions does it return a list of matches, or a match

1.   List item
2.   List item

object. you want a match object look up pandas.apply. once you have a match item, the possibilities are endless :) match.groups gives you all the group.

match.start (where it should start)
match.end(where it should end)